In [26]:
import subprocess
import sys
import random as rd
import webbrowser

def instalar(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
except ImportError:
    instalar('pandas')
    import pandas as pd

In [27]:
#Criar backup
try:
    backup = pd.read_csv('save.csv')
except:
    backup = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
backup.to_csv('save_backup.csv', index=False)

In [28]:
def escolhe_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    for index, row in data.iterrows(): 
        print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')

In [29]:
def ver_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
    except:
        print('Você ainda não tem nenhuma carta na coleção! Abra alguns pacotes para começar a colecionar!')

In [30]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity'])

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [31]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)

    return box

In [32]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])

    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)

    save = atualizar_save(save, box)
    save.sort_values(by=['set', 'id'], inplace=True)
    save.to_csv('save.csv', index=False)
    
    return box

In [33]:
def gerar_link_wiki(nome_carta):
    # A wiki usa underscores no lugar de espaços
    nome_formatado = nome_carta.replace(' ', '_')
    return f'https://cardfight.fandom.com/wiki/{nome_formatado}'

In [34]:
def web_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            link = gerar_link_wiki(row['name'])
            print(f'Abrindo Wiki de: {row["name"]}...')
            webbrowser.open(link) # Isso abre o navegador automaticamente
    except:
        print('Erro ao carregar coleção.')

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def extrair_vanguard_final():
    url = "https://cardfight.fandom.com/wiki/Booster_Set_3:_Demonic_Lord_Invasion"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print("Buscando dados...")
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    cards_data = []

    # O site da Fandom usa tabelas dentro de classes 'wikitable' 
    # mas o conteúdo é separado por idiomas/versões. 
    # Vamos pegar TODAS as linhas de TODAS as tabelas e filtrar depois.
    rows = soup.find_all('tr')

    for row in rows:
        cols = row.find_all(['td', 'th'])
        
        # Se a linha tiver pelo menos 5 colunas, pode ser uma carta
        if len(cols) >= 5:
            card_id = cols[0].get_text(strip=True)
            
            # Filtro rigoroso: O ID precisa ter uma barra '/' (ex: BT03/001)
            if "/" in card_id and any(char.isdigit() for char in card_id):
                try:
                    name = cols[1].get_text(strip=True)
                    grade = cols[2].get_text(strip=True)
                    clan = cols[3].get_text(strip=True)
                    # Algumas tabelas da wiki mesclam colunas, tratamos isso aqui:
                    card_type = cols[4].get_text(strip=True) if len(cols) > 4 else ""
                    rarity = cols[5].get_text(strip=True) if len(cols) > 5 else ""

                    cards_data.append({
                        "set": card_id.split('/')[0],
                        "id": card_id,
                        "name": name,
                        "grade": grade,
                        "clan": clan,
                        "type": card_type,
                        "rarity": rarity
                    })
                except IndexError:
                    continue

    if cards_data:
        df = pd.DataFrame(cards_data)
        # Remove duplicatas (cartas que aparecem na lista JPN e EN ao mesmo tempo)
        df = df.drop_duplicates(subset=['id', 'rarity'])
        
        df.to_csv("vanguard_bt03.csv", index=False, encoding='utf-8-sig')
        print(f"Pronto! Encontrei {len(df)} cartas.")
    else:
        # Se falhar, vamos imprimir o que o script está vendo para depurar
        print("Ainda não encontrei nada. Verificando estrutura...")
        sample_rows = len(rows)
        print(f"O script analisou {sample_rows} linhas no total, mas nenhuma passou no filtro de ID.")

extrair_vanguard_final()

Buscando dados...
Ainda não encontrei nada. Verificando estrutura...
O script analisou 0 linhas no total, mas nenhuma passou no filtro de ID.
